### 1. Import Libraries
Load spatial libraries. Notice we imported `voronoi_diagram` from Shapely!

In [62]:
import pandas as pd
import geopandas as gpd
from shapely.ops import voronoi_diagram
from shapely.geometry import MultiPoint
import io
import warnings

warnings.filterwarnings('ignore')

### 2. Load the 2026 Tournament Data
*Updated: Added the `Mascot` column.*
*Updated: Added the First Four! The 64 advancing teams have a Furthest_Round of 64. The 4 teams that lose in the First Four start with a Furthest_Round of 68.*

In [70]:
csv_data = """School,Mascot,Arena,Hex,Hex2,Lat,Lon,Furthest_Round,Capacity,Avg_Attendance
Duke,Blue Devils,Cameron Indoor Stadium,#003087,#FFFFFF,35.9973,-78.9423,64,9314,9314
Arizona,Wildcats,McKale Center,#CC0033,#003366,32.2300,-110.9450,64,14644,14058
Michigan,Wolverines,Crisler Center,#00274C,#FFCB05,42.2657,-83.7469,64,12707,11500
Florida,Gators,Exactech Arena,#0021A5,#FA4616,29.6496,-82.3512,64,10151,10000
Houston,Cougars,Fertitta Center,#C8102E,#FFFFFF,29.7153,-95.3406,64,7035,7035
UConn,Huskies,Gampel Pavilion,#000E2F,#FFFFFF,41.8055,-72.2523,64,10167,10167
Iowa State,Cyclones,Hilton Coliseum,#C8102E,#F1BE48,42.0219,-93.6373,64,14267,14062
Purdue,Boilermakers,Mackey Arena,#CEB888,#000000,40.4333,-86.9158,64,14876,14876
Michigan State,Spartans,Breslin Center,#18453B,#FFFFFF,42.7282,-84.4849,64,14797,14797
Illinois,Fighting Illini,State Farm Center,#E84A27,#13294B,40.0962,-88.2356,64,15544,15091
Gonzaga,Bulldogs,McCarthey Athletic Center,#041E42,#C1C6C8,47.6661,-117.3995,64,6000,6000
Virginia,Cavaliers,John Paul Jones Arena,#232D4B,#F84C1E,38.0464,-78.5135,64,14593,13800
Nebraska,Cornhuskers,Pinnacle Bank Arena,#E41C38,#FDF2D9,40.8175,-96.7119,64,15000,14500
Alabama,Crimson Tide,Coleman Coliseum,#9E1B32,#828A8F,33.2017,-87.5385,64,15383,11500
Kansas,Jayhawks,Allen Fieldhouse,#0051BA,#E8000D,38.9543,-95.2528,64,16300,16300
Arkansas,Razorbacks,Bud Walton Arena,#9D2235,#FFFFFF,36.0593,-94.1783,64,19200,19000
Vanderbilt,Commodores,Memorial Gymnasium,#866D4B,#000000,36.1432,-86.8021,64,14316,6500
St. John's,Red Storm,Carnesecca Arena,#BA0C2F,#FFFFFF,40.7244,-73.8090,64,5602,10000
Texas Tech,Red Raiders,United Supermarkets Arena,#CC0000,#000000,33.5855,-101.8895,64,15098,14000
Wisconsin,Badgers,Kohl Center,#C5050C,#FFFFFF,43.0706,-89.3980,64,17287,15006
Tennessee,Volunteers,Thompson-Boling Arena,#FF8200,#FFFFFF,35.9529,-83.9242,64,21678,20026
North Carolina,Tar Heels,Dean Smith Center,#7BAFD4,#FFFFFF,35.8996,-79.0435,64,21750,20593
Louisville,Cardinals,KFC Yum! Center,#AD0000,#000000,38.2575,-85.7515,64,22090,14864
BYU,Cougars,Marriott Center,#002255,#FFFFFF,40.2520,-111.6508,64,18987,15500
Kentucky,Wildcats,Rupp Arena,#0033A0,#FFFFFF,38.0494,-84.5025,64,20500,19900
Saint Mary's,Gaels,UCU Pavilion,#E41C38,#002B5C,37.8427,-122.1065,64,3500,3300
Miami (FL),Hurricanes,Watsco Center,#F47321,#005030,25.7176,-80.2790,64,7972,6000
UCLA,Bruins,Pauley Pavilion,#2D68C4,#F2A900,34.0704,-118.4468,64,13800,8500
Clemson,Tigers,Littlejohn Coliseum,#F56600,#522D80,34.6811,-82.8443,64,9000,7500
Villanova,Wildcats,Finneran Pavilion,#00205B,#13B5EA,40.0385,-75.3426,64,6501,6500
Ohio State,Buckeyes,Value City Arena,#CE1126,#666666,40.0076,-83.0227,64,19049,14000
Georgia,Bulldogs,Stegeman Coliseum,#BA0C2F,#000000,33.9455,-83.3731,64,10523,8000
Utah State,Aggies,Dee Glen Smith Spectrum,#0F2439,#9D968D,41.7431,-111.8105,64,10270,9000
TCU,Horned Frogs,Schollmaier Arena,#4D1979,#FFFFFF,32.7090,-97.3670,64,6800,6000
Saint Louis,Billikens,Chaifetz Arena,#003DA5,#FFFFFF,38.6346,-90.2307,64,10600,7000
Iowa,Hawkeyes,Carver-Hawkeye Arena,#FFCD00,#000000,41.6632,-91.5513,64,15056,12000
Santa Clara,Broncos,Leavey Center,#B30738,#FFFFFF,37.3514,-121.9388,64,4500,2000
UCF,Knights,Addition Financial Arena,#000000,#BA9B37,28.6075,-81.1965,64,10000,6000
Missouri,Tigers,Mizzou Arena,#F1B82D,#000000,38.9285,-92.3332,64,15061,11000
Texas A&M,Aggies,Reed Arena,#500000,#FFFFFF,30.6056,-96.3458,64,12989,9500
NC State,Wolfpack,PNC Arena,#CC0000,#000000,35.8033,-78.7218,68,19722,13000
Texas,Longhorns,Moody Center,#BF5700,#FFFFFF,30.2818,-97.7315,64,10763,10500
SMU,Mustangs,Moody Coliseum,#0033A0,#CC0000,32.8407,-96.7845,68,7000,4500
Miami (OH),RedHawks,Millett Hall,#B61E2E,#FFFFFF,39.5160,-84.7335,64,6400,2000
VCU,Rams,Siegel Center,#FFB300,#000000,37.5544,-77.4528,64,7637,7637
South Florida,Bulls,Yuengling Center,#006747,#CFC493,28.0645,-82.4116,64,10411,5500
McNeese,Cowboys,Legacy Center,#00447C,#FFD100,30.1770,-93.2138,64,4242,3500
Akron,Zips,James A. Rhodes Arena,#041E42,#A8996E,41.0722,-81.5113,64,5500,2500
UNI,Panthers,McLeod Center,#4B116F,#FFCC00,42.5126,-92.4659,64,6650,4000
High Point,Panthers,Qubein Center,#330072,#FFFFFF,35.9754,-79.9961,64,4500,4000
California Baptist,Lancers,CBU Events Center,#003E7E,#BFA56A,33.9317,-117.4246,64,5050,3000
Hofstra,Pride,Mack Sports Complex,#003594,#FFB300,40.7185,-73.5986,64,5023,2000
Troy,Trojans,Trojan Arena,#8A2432,#8A8D8F,31.7995,-85.9550,64,5200,2500
Hawaii,Rainbow Warriors,Stan Sheriff Center,#024731,#FFFFFF,21.2941,-157.8184,64,10300,5500
North Dakota State,Bison,Scheels Center,#005643,#FFC72C,46.9037,-96.8049,64,5700,3000
Penn,Quakers,The Palestra,#990000,#001A57,39.9524,-75.1888,64,8722,4000
Wright State,Raiders,Nutter Center,#026937,#CEB888,39.7801,-84.0531,64,10400,4500
Kennesaw State,Owls,Convocation Center,#FEB81C,#000000,34.0378,-84.5807,64,3805,2000
Tennessee State,Tigers,Gentry Complex,#0033A0,#FFFFFF,36.1687,-86.8284,64,10500,2500
Idaho,Vandals,ICCU Arena,#CC9900,#000000,46.7266,-117.0165,64,4200,2000
Furman,Paladins,Timmons Arena,#582C83,#FFFFFF,34.9254,-82.4371,64,2500,1800
Queens,Royals,Curry Arena,#00205B,#B9975B,35.1895,-80.8315,64,2500,1500
Siena,Saints,MVP Arena,#006532,#F2A900,42.6486,-73.7547,64,15229,6000
Prairie View A&M,Panthers,William Nicks Building,#4B2682,#F1B82D,30.0934,-95.9897,64,5520,1500
LIU,Sharks,Steinberg Wellness Center,#69B3E7,#FFC72C,40.6905,-73.9806,64,2500,1000
Howard,Bison,Burr Gymnasium,#E8000D,#003A63,38.9238,-77.0223,64,2700,1500
UMBC,Retrievers,Chesapeake Employers Insurance Arena,#FFC20E,#000000,39.2526,-76.7076,68,4722,2000
Lehigh,Mountain Hawks,Stabler Arena,#502D16,#FFFFFF,40.5977,-75.3562,68,5600,1500"""

all_teams_df = pd.read_csv(io.StringIO(csv_data))

# Upgrade all remaining 64 teams to "Assumed Champions" (Furthest_Round = 1)
all_teams_df.loc[all_teams_df['Furthest_Round'] == 64, 'Furthest_Round'] = 1

print(f"Loaded {len(all_teams_df)} teams and set base to Assumed Champions.")

Loaded 68 teams and set base to Assumed Champions.


### 3. Load the US Boundary
Read a US States GeoJSON directly from a web URL, dissolve the state borders into a single national polygon, and clip it to the contiguous lower 48 states.

In [64]:
print("Fetching 50-State US Boundary from the web...")

# 1. Define our new Web-friendly CRS for all 50 states
full_us_epsg = 3857 

# 2. Read US states directly from a public GeoJSON URL
url = "https://raw.githubusercontent.com/PublicaMundi/MappingAPI/master/data/geojson/us-states.json"
states = gpd.read_file(url)

# 3. Filter out Puerto Rico (id '72') to leave just the 50 states + DC
states = states[states['id'] != '72']

# 4. Dissolve all the individual states into one giant national boundary
us_full = states.dissolve()

# 5. Project to Web Mercator
us_boundary = us_full.to_crs(epsg=full_us_epsg)

print("50-State Boundary ready!")

Fetching 50-State US Boundary from the web...
50-State Boundary ready!


### 4. The Smart Voronoi GeoJSON Generator
This dynamically draws the territories, checks adjacent teams for color clashes using RGB Euclidean distance, and assigns dynamic borders before clipping to the US shape.

In [65]:
import math

def export_round_geojson(target_round):
    print(f"Processing Round of {target_round}...")
    
    active_teams = all_teams_df[all_teams_df['Furthest_Round'] <= target_round].copy()
    active_teams.reset_index(drop=True, inplace=True)
    
    # Notice we are now using full_us_epsg (3857)
    teams_gdf = gpd.GeoDataFrame(
        active_teams, 
        geometry=gpd.points_from_xy(active_teams.Lon, active_teams.Lat), 
        crs="EPSG:4326"
    ).to_crs(epsg=full_us_epsg)

    team_points = MultiPoint(teams_gdf.geometry.values)
    voronoi_polys = voronoi_diagram(team_points)
    voronoi_gdf = gpd.GeoDataFrame(geometry=list(voronoi_polys.geoms), crs=full_us_epsg)

    territories = gpd.sjoin(
        voronoi_gdf, 
        teams_gdf[['School', 'Mascot', 'Hex', 'Hex2', 'Arena', 'Capacity', 'Avg_Attendance', 'geometry']], 
        how='inner', predicate='intersects'
    )

    # We clip to the new 50-state boundary here
    final_territories = gpd.clip(territories, us_boundary)
    final_territories = final_territories.reset_index(drop=True)

    final_territories['geometry'] = final_territories['geometry'].simplify(2000)
    final_territories = final_territories.to_crs(epsg=4326)
    
    filename = f"round_{target_round}.geojson"
    final_territories.to_file(filename, driver="GeoJSON")
    print(f"Saved {filename} with 50 states!")

In [66]:
def eliminate_teams(losing_teams, round_eliminated):
    """
    Updates the dataset to reflect teams eliminated in a specific round.
    """
    success_count = 0
    for team in losing_teams:
        if team in all_teams_df['School'].values:
            # Lock the losing team's status to the round they lost in
            all_teams_df.loc[all_teams_df['School'] == team, 'Furthest_Round'] = round_eliminated
            success_count += 1
        else:
            print(f"⚠️ Typo Alert: Could not find '{team}' in the dataset. Check your spelling!")
            
    print(f"✅ Successfully eliminated {success_count} teams in the Round of {round_eliminated}!")

In [71]:
# List the teams that lost their Round of 64 matchup today
round_of_64_losers = [
    'Siena',
    'Ohio State',
    'South Florida',
    'North Dakota State',
    'McNeese',
    'Troy',
    'North Carolina',
    'Penn',
    "Saint Mary's",
    'Idaho',
    'Wisconsin',
    'Hawaii',
    'BYU',
    'Kennesaw State',
    'Howard',
    'Georgia'
]

# Eliminate them in the Round of 64
eliminate_teams(round_of_64_losers, 64)

# Generate the brand new Round of 32 map!
export_round_geojson(32)

✅ Successfully eliminated 16 teams in the Round of 64!
Processing Round of 32...
Saved round_32.geojson with 50 states!


### 5. Execute Pipeline
Run the generator for all rounds, now including 68!

In [72]:
# Notice 68 is now included at the start!
rounds = [68, 64, 32]

for r in rounds:
    export_round_geojson(r)
    
print("\nAll perfect Voronoi maps generated successfully!")

Processing Round of 68...
Saved round_68.geojson with 50 states!
Processing Round of 64...
Saved round_64.geojson with 50 states!
Processing Round of 32...
Saved round_32.geojson with 50 states!

All perfect Voronoi maps generated successfully!


### 6. Interactive Preview using Folium
Render the generated Voronoi GeoJSON directly in the notebook. This now uses the `Hex` column for the fill and the `Hex2` column for the thicker borders to separate adjacent teams!

In [74]:
import folium
import json

m = folium.Map(location=[39.8283, -98.5795], zoom_start=4, tiles='CartoDB positron')

with open('round_64.geojson', 'r') as f:
    geojson_data = json.load(f)

folium.GeoJson(
    geojson_data,
    style_function=lambda feature: {
        'fillColor': feature['properties']['Hex'],
        'fillOpacity': 0.8,
        
        # --- Active styling: Clean white borders ---
        'color': '#ffffff', 
        'weight': 1.5
        
        # --- Commented out: Smart border styling ---
        # 'color': feature['properties'].get('Border_Color', '#ffffff'), 
        # 'weight': feature['properties'].get('Border_Weight', 1.5)
    },
    tooltip=folium.GeoJsonTooltip(
        fields=['School', 'Mascot', 'Arena', 'Capacity', 'Avg_Attendance'],
        aliases=['School:', 'Mascot:', 'Arena:', 'Capacity:', 'Avg Attendance:'],
        localize=True,
        style=("background-color: white; color: #333333; font-family: arial; font-size: 14px; padding: 10px;")
    )
).add_to(m)

m